# Soluzioni — Lezione 1: Python da zero, regressione lineare e logistica
[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/main/ml-humanities/solutions/01_python_da_zero_regressione_soluzioni.ipynb)

Ci sono molte soluzioni giuste: queste sono solo una possibilità.

In [1]:
# ⚙️ CELLA DI PREPARAZIONE — eseguitela per prima (Shift+Invio) e non preoccupatevi del codice.
# Cerca la cartella "data" del corso; se non la trova (per esempio su Google Colab),
# scarica il materiale del corso da GitHub. Funziona anche senza internet, se avete la cartella.
from pathlib import Path
import subprocess

def trova_dati():
    for p in [Path("../data"), Path("data"), Path("ml-humanities/data"), Path("yuribizzoni/ml-humanities/data")]:
        if (p / "lessico_felicita.csv").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuri-bizzoni/yuribizzoni"], check=True)
    return Path("yuribizzoni/ml-humanities/data")

DATI = trova_dati()
print("Dati trovati in:", DATI.resolve())

Dati trovati in: /home/user/yuribizzoni/ml-humanities/data


In [2]:
import re
import pandas as pd

def parole(testo):
    """Trasforma un testo in una lista di parole minuscole."""
    return re.findall(r"\w+", testo.lower())

famiglie = pd.read_csv(DATI / "famiglie_felicita_it.csv")
FORME_FELICI = set(famiglie["forma"])
print(len(FORME_FELICI), "forme felici, per esempio:", sorted(FORME_FELICI)[:8])

35 forme felici, per esempio: ['allegra', 'allegramente', 'allegre', 'allegrezza', 'allegri', 'allegria', 'allegro', 'beata']


## Esercizio 1 — Una parola alla volta

In [3]:
for parola in ["lieta", "dolore", "beati"]:
    print(parola, "→", len(parola), "lettere; prima:", parola[0], "ultima:", parola[-1])
    if parola in FORME_FELICI:
        print("   è una forma felice")
    else:
        print("   non è nel nostro elenco di forme felici")

lieta → 5 lettere; prima: l ultima: a
   è una forma felice
dolore → 6 lettere; prima: d ultima: e
   non è nel nostro elenco di forme felici
beati → 5 lettere; prima: b ultima: i
   è una forma felice


## Esercizio 2 — La poesia più felice

In [4]:
canti = pd.read_csv(DATI / "leopardi_canti.csv")
canti["n_parole"] = canti["testo"].apply(lambda t: len(parole(t)))
canti["n_felici"] = canti["testo"].apply(lambda t: sum(p in FORME_FELICI for p in parole(t)))
canti["tasso_felicita"] = canti["n_felici"] / canti["n_parole"] * 1000
top = canti.sort_values("tasso_felicita", ascending=False).head(5)
top[["titolo", "n_parole", "n_felici", "tasso_felicita"]]

,titolo,n_parole,n_felici,tasso_felicita
24,Il sabato del villaggio,296,4,13.513514
16,Consalvo,1054,14,13.282732
4,A un vincitore nel pallone,410,5,12.195122
10,Il passero solitario,352,4,11.363636
23,La quiete dopo la tempesta,293,3,10.238908


In [5]:
print(top["titolo"].iloc[0])
print(top["testo"].iloc[0][:500])

Il sabato del villaggio
La donzelletta vien dalla campagna,
In sul calar del sole,
Col suo fascio dell'erba; e reca in mano
Un mazzolin di rose e di viole,
Onde, siccome suole,
Ornare ella si appresta
Dimani, al dì di festa, il petto e il crine.
Siede con le vicine
Su la scala a filar la vecchierella,
Incontro là dove si perde il giorno;
E novellando vien del suo buon tempo,
Quando ai dì della festa ella si ornava,
Ed ancor sana e snella
Solea danzar la sera intra di quei
Ch'ebbe compagni dell'età più bella.
Già tutta 


**Commento.** In testa troviamo *Il sabato del villaggio* e *La quiete dopo la tempesta*: due poesie che parlano di felicità
proprio per dire che è un'attesa o una pausa dal dolore ("Piacer figlio d'affanno", si legge nella *Quiete*).
Il conteggio misura il **lessico**, non lo stato d'animo: ironia, negazione e nostalgia gli sfuggono.
Notate anche che le poesie brevi hanno tassi più "estremi": poche parole felici in un testo corto pesano molto.

## Esercizio 3 — Leopardi e Dante

In [6]:
from sklearn.linear_model import LinearRegression
canti["n_tipi"] = canti["testo"].apply(lambda t: len(set(parole(t))))
reg_leo = LinearRegression().fit(canti[["n_parole"]], canti["n_tipi"])

dc = pd.read_csv(DATI / "commedia_canti.csv")
dc["n_parole"] = dc["testo"].apply(lambda t: len(parole(t)))
dc["n_tipi"] = dc["testo"].apply(lambda t: len(set(parole(t))))
reg_dante = LinearRegression().fit(dc[["n_parole"]], dc["n_tipi"])

print("Pendenza Leopardi:", round(reg_leo.coef_[0], 3), " R²:", round(reg_leo.score(canti[["n_parole"]], canti["n_tipi"]), 3))
print("Pendenza Dante:   ", round(reg_dante.coef_[0], 3), " R²:", round(reg_dante.score(dc[["n_parole"]], dc["n_tipi"]), 3))

Pendenza Leopardi: 0.489  R²: 0.987
Pendenza Dante:    0.36  R²: 0.625


**Commento.** La pendenza dice quante parole *nuove* si aggiungono, in media, per ogni parola in più di testo.
Attenzione al confronto: i canti di Dante hanno tutti lunghezze simili, mentre le poesie di
Leopardi vanno da poche decine a quasi duemila parole. E il vocabolario non cresce in modo lineare: più un testo è lungo,
più le parole si ripetono (è la *legge di Heaps*). Una retta è un'approssimazione utile solo in un intervallo limitato.